<a href="https://colab.research.google.com/github/mixxr/gcolab-ml/blob/main/funnel/09_admet_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Episode 9 — ADMET & Optimization
**Drug Discovery in Code** · [Life Sciences AI Applications]
A potent binder is **not** a drug. Most candidates die later on **ADMET** — Absorption,
Distribution, Metabolism, Excretion, and Toxicity. Today we score molecules for drug-likeness and
reactive/toxic liabilities, then do the core move of medicinal chemistry: spot a problem group and
**fix it**, watching the numbers improve.
> Educational only — not medical advice.

## Three quick, code-able ADMET signals
- **QED** — Quantitative Estimate of Drug-likeness, 0→1 (higher is more drug-like).
- **SA score** — synthetic accessibility, ~1 (easy) → 10 (hard to make).
- **Brenk alerts** — a catalog of reactive/toxic/unstable substructures to avoid.

In [ ]:
import sys
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, Draw, RDConfig
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

# RDKit ships a synthetic-accessibility scorer in its contrib dir
sys.path.append(RDConfig.RDContribDir + "/SA_Score")
import sascorer

params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
brenk = FilterCatalog(params)

## 1. Profile a few molecules

In [ ]:
def profile(smiles):
    m = Chem.MolFromSmiles(smiles)
    hits = [e.GetDescription() for e in brenk.GetMatches(m)]
    return dict(
        MolWt  = round(Descriptors.MolWt(m), 1),
        QED    = round(QED.qed(m), 2),
        SA     = round(sascorer.calculateScore(m), 2),
        alerts = ", ".join(hits) or "none",
    )

profile('COc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OCCCN1CCOCC1')  # gefitinib

## 2. Find a liability
Here's a hypothetical hit — same quinazoline scaffold, but someone hung a **nitro group** on the
aniline. Nitro aromatics are a classic mutagenicity flag.

In [ ]:
hit = 'COc1cc2ncnc(Nc3cccc([N+](=O)[O-])c3)c2cc1OC'
profile(hit)   # -> alerts: 'nitro_group, ...'

## 3. Optimise — swap the offending group
The medicinal-chemistry move: replace the nitro with a benign **chloro** (a common, metabolically
stable substituent) and re-profile.

In [ ]:
fixed = 'COc1cc2ncnc(Nc3cccc(Cl)c3)c2cc1OC'
profile(fixed)   # -> alerts: 'none'

In [ ]:
before, after = Chem.MolFromSmiles(hit), Chem.MolFromSmiles(fixed)
Draw.MolsToGridImage([before, after], molsPerRow=2, subImgSize=(400,340),
    legends=[f"HIT  QED {QED.qed(before):.2f}", f"FIXED  QED {QED.qed(after):.2f}"])

One atom-group swap: the **nitro alert disappears**, QED climbs from **0.57 → 0.79**, and the
molecule even gets *easier* to synthesise. That's optimization in miniature — keep the part that
binds, remove the part that would have killed the program. Real campaigns do this hundreds of
times, balancing potency against ADMET at every step.

## Recap
- **Potency ≠ drug** — most candidates fail on ADMET, so screen for it early.
- **QED**, **SA score**, and **Brenk alerts** are fast, code-able first-pass filters.
- **Optimise** by excising liability groups and re-scoring — potency in, problems out.

**Next — Episode 10:** the **capstone** — chain the whole pipeline end-to-end into one
reproducible mini virtual-screening campaign against EGFR.